# NEXUS — Google Colab + GPU + Ollama + Google Drive

Este notebook adapta o projeto **`overcyber/minicurso-mult-agents`** para Google Colab, mantendo a progressão pedagógica original em cinco dias.

## Objetivos desta versão

- executar com **GPU NVIDIA do Colab**;
- instalar e executar **Ollama dentro do Colab**;
- persistir os modelos Ollama no **Google Drive**, evitando novo download a cada runtime;
- persistir cache do **Hugging Face** no Google Drive;
- manter uma cópia persistente do **código** no Google Drive;
- executar código a partir de `/content`, onde SQLite/Chroma funcionam melhor;
- sincronizar `.chroma`, `.chroma_hf`, bancos SQLite, relatórios, traces e logs com o Drive;
- separar explicitamente **Dia 1, Dia 2, Dia 3, Dia 4 e Dia 5**;
- executar a bateria de **55 casos pytest** depois da instalação;
- deixar claro o que é código original e o que é adaptação específica para Colab/GPU.

## Estado de validação do material original

- **Dia 1:** já havia sido testado antes desta adaptação.
- **Dias 2–5:** devem ser tratados como experimentais até serem executados ponta a ponta no Colab.
- A revisão encontrou lacunas de integração nos Dias 3–5; elas são destacadas nas seções correspondentes.

> **Antes de executar:** em **Runtime → Change runtime type**, selecione uma GPU.

## 0. Arquitetura de persistência

O notebook usa duas estratégias diferentes:

```text
Google Drive
├── repo/                     cópia persistente do GitHub
├── ollama/models/            blobs e manifests do Ollama
├── huggingface/              cache HF/Transformers/Sentence Transformers
└── state/                    Chroma, SQLite, saídas, traces e logs
             ⇅ sincronização
/content/minicurso-mult-agents
└── codigo/nexus/             execução rápida no disco local do Colab
```

**Motivo:** modelos e caches podem ficar diretamente no Drive. Já SQLite e Chroma fazem muitas operações pequenas e transacionais; a execução local em `/content` é mais adequada, com cópia periódica do estado para o Drive.

In [1]:
# 0.1 — Montar o Google Drive e definir diretórios persistentes

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

from pathlib import Path
import os, sys, shutil, subprocess, time, json, re, uuid, importlib

DRIVE_ROOT = Path("/content/drive/MyDrive/minicurso-mult-agents-colab")
DRIVE_REPO = DRIVE_ROOT / "repo"
DRIVE_STATE = DRIVE_ROOT / "state"
OLLAMA_MODELS_DIR = DRIVE_ROOT / "ollama" / "models"
HF_ROOT = DRIVE_ROOT / "huggingface"

RUNTIME_REPO = Path("/content/minicurso-mult-agents")
NEXUS = RUNTIME_REPO / "codigo" / "nexus"

for p in [DRIVE_ROOT, DRIVE_STATE, OLLAMA_MODELS_DIR, HF_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

print("DRIVE_ROOT =", DRIVE_ROOT)
print("RUNTIME_REPO =", RUNTIME_REPO)

Mounted at /content/drive
DRIVE_ROOT = /content/drive/MyDrive/minicurso-mult-agents-colab
RUNTIME_REPO = /content/minicurso-mult-agents


In [2]:
# 0.2 — Verificar a GPU antes de baixar modelos

def executar(cmd, *, check=True, capture=False, env=None, cwd=None):
    if isinstance(cmd, str):
        return subprocess.run(
            cmd, shell=True, check=check, text=True,
            capture_output=capture, env=env, cwd=cwd
        )
    return subprocess.run(
        cmd, check=check, text=True,
        capture_output=capture, env=env, cwd=cwd
    )

gpu = executar(
    ["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
     "--format=csv,noheader"],
    check=False, capture=True,
)

if gpu.returncode != 0:
    raise RuntimeError(
        "GPU NVIDIA não detectada. No Colab, selecione Runtime > "
        "Change runtime type > GPU e execute novamente."
    )

print("GPU:", gpu.stdout.strip())
executar(["nvidia-smi"], check=False)

GPU: Tesla T4, 15360 MiB, 580.82.07


CompletedProcess(args=['nvidia-smi'], returncode=0)

## 1. Código persistente: GitHub → Drive → `/content`

A cópia em `repo/` no Drive é persistente. Em cada sessão o notebook cria uma cópia limpa em `/content` para execução.

Se houver modificações locais no clone persistente do Drive, o notebook **não faz `pull` automaticamente**, para não sobrescrever trabalho do aluno.

In [3]:
# 1.1 — Clonar/atualizar o repositório no Drive

REPO_URL = "https://github.com/overcyber/minicurso-mult-agents.git"
REPO_REF = "main"

if not (DRIVE_REPO / ".git").exists():
    DRIVE_REPO.parent.mkdir(parents=True, exist_ok=True)
    executar(["git", "clone", REPO_URL, str(DRIVE_REPO)])
else:
    status = executar(
        ["git", "-C", str(DRIVE_REPO), "status", "--porcelain"],
        capture=True
    ).stdout.strip()
    if status:
        print("ATENÇÃO: há alterações locais no clone persistente do Drive.")
        print("Nenhum pull automático será executado.")
        print(status[:3000])
    else:
        executar(["git", "-C", str(DRIVE_REPO), "fetch", "origin", REPO_REF])
        executar(["git", "-C", str(DRIVE_REPO), "checkout", REPO_REF])
        executar(["git", "-C", str(DRIVE_REPO), "pull", "--ff-only", "origin", REPO_REF])

print(
    executar(
        ["git", "-C", str(DRIVE_REPO), "log", "-1", "--oneline"],
        capture=True
    ).stdout.strip()
)

36ef3d7 Colab GPU: Ollama + Drive persistente + Dias 1–5


In [4]:
# 1.2 — Criar cópia de execução em /content

if RUNTIME_REPO.exists():
    shutil.rmtree(RUNTIME_REPO)

IGNORE = shutil.ignore_patterns(
    ".git", ".venv", "venv", "__pycache__", "*.pyc",
    ".chroma", ".chroma_hf", "*.db", "*.db-wal", "*.db-shm",
    "saida", "tracos"
)

shutil.copytree(DRIVE_REPO, RUNTIME_REPO, ignore=IGNORE)

if not NEXUS.exists():
    raise RuntimeError(f"Projeto NEXUS não encontrado em {NEXUS}")

os.chdir(NEXUS)
print("Executando a partir de:", Path.cwd())

Executando a partir de: /content/minicurso-mult-agents/codigo/nexus


## 2. Dependências reproduzíveis no Colab

O `requirements.txt` original usa limites inferiores amplos. Para reduzir deriva do ambiente, esta célula restringe alguns componentes centrais às versões atuais verificadas em setembro de 2026 e deixa o restante ser resolvido pelo `pip`.

A célula também instala `accelerate`, necessário para os exemplos Hugging Face que usam `device_map`.

In [5]:
# 2.1 — Instalar dependências

constraints = Path("/content/nexus-colab-constraints.txt")
constraints.write_text(
    "\n".join([
        "langchain==1.4.0",
        "langgraph==1.2.11",
        "langgraph-checkpoint-sqlite==3.1.1",
        "langchain-ollama==1.1.0",
        "langchain-chroma==1.1.0",
        "chromadb==1.5.9",
    ]) + "\n",
    encoding="utf-8",
)

requirements = NEXUS / "requirements.txt"

executar([
    sys.executable, "-m", "pip", "install", "-q",
    "-r", str(requirements),
    "-c", str(constraints),
    "accelerate>=1.0",
])

executar([sys.executable, "-m", "pip", "check"], check=False)
print("Dependências instaladas.")

Dependências instaladas.


In [6]:
# 2.2 — Cache Hugging Face persistente no Google Drive

os.environ["HF_HOME"] = str(HF_ROOT)
os.environ["HF_HUB_CACHE"] = str(HF_ROOT / "hub")
os.environ["TRANSFORMERS_CACHE"] = str(HF_ROOT / "transformers")
os.environ["SENTENCE_TRANSFORMERS_HOME"] = str(HF_ROOT / "sentence-transformers")

for p in [
    Path(os.environ["HF_HUB_CACHE"]),
    Path(os.environ["TRANSFORMERS_CACHE"]),
    Path(os.environ["SENTENCE_TRANSFORMERS_HOME"]),
]:
    p.mkdir(parents=True, exist_ok=True)

print("HF_HOME =", os.environ["HF_HOME"])

HF_HOME = /content/drive/MyDrive/minicurso-mult-agents-colab/huggingface


## 3. Restaurar e salvar estado

Os itens abaixo são sincronizados entre `/content` e o Drive:

- `.chroma`;
- `.chroma_hf`;
- `.chroma_hf_gpu`;
- bancos SQLite `*.db`, `*.db-wal`, `*.db-shm`;
- `saida/`;
- `tracos/`;
- `ollama.log`;
- CSVs e relatórios gerados na raiz do NEXUS.

Execute `persistir_estado()` depois de uma etapa importante e sempre antes de encerrar o runtime.

In [7]:
# 3.1 — Funções de persistência

DIRS_ESTADO = [".chroma", ".chroma_hf", ".chroma_hf_gpu", "saida", "tracos"]
GLOBS_ESTADO = ["*.db", "*.db-wal", "*.db-shm", "*.csv", "*.json", "*.jsonl", "*.md", "ollama.log"]

def _copiar_item(src: Path, dst: Path):
    if not src.exists():
        return
    if src.is_dir():
        if dst.exists():
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
    else:
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)

def restaurar_estado():
    for nome in DIRS_ESTADO:
        _copiar_item(DRIVE_STATE / nome, NEXUS / nome)

    for pattern in ["*.db", "*.db-wal", "*.db-shm", "*.csv", "*.json", "*.jsonl", "ollama.log"]:
        for src in DRIVE_STATE.glob(pattern):
            _copiar_item(src, NEXUS / src.name)

    print("Estado restaurado de", DRIVE_STATE)

def persistir_estado():
    DRIVE_STATE.mkdir(parents=True, exist_ok=True)

    for nome in DIRS_ESTADO:
        _copiar_item(NEXUS / nome, DRIVE_STATE / nome)

    for pattern in GLOBS_ESTADO:
        for src in NEXUS.glob(pattern):
            # Não copiamos os Markdown-fonte do curso como "estado".
            if pattern == "*.md" and src.name in {"README.md"}:
                continue
            _copiar_item(src, DRIVE_STATE / src.name)

    print("Estado persistido em", DRIVE_STATE)

restaurar_estado()

Estado restaurado de /content/drive/MyDrive/minicurso-mult-agents-colab/state


## 4. Ollama no Colab com GPU e modelos persistentes

O Ollama será instalado no runtime Linux do Colab. O diretório de modelos, porém, aponta para o Google Drive por meio de `OLLAMA_MODELS`.

Modelos básicos do curso:

- `qwen3:4b` — modelo principal;
- `nomic-embed-text` — embeddings do RAG.

No Dia 5 o código original também referencia `qwen3:1.7b` e `qwen3:8b`. Para evitar downloads e uso de VRAM desnecessários, este notebook usa `qwen3:4b` para todos os papéis por padrão. Há uma célula opcional para baixar os três tamanhos.

In [12]:
import subprocess
import os

# 4.0 — Diagnóstico e instalação robusta do Ollama (colocado aqui para rodar antes de 4.1)

# A célula 4.1 tenta instalar o Ollama. Se houver falha (exit status 1),
# pode ser devido a dependências ou problemas de ambiente.
# Este bloco tenta garantir que as ferramentas básicas (curl, tar, gzip, zstd)
# estejam atualizadas e presentes, e oferece uma instalação alternativa se necessário.

# Garante que apt-get esteja atualizado e instala ferramentas essenciais...
print("Atualizando apt-get e instalando ferramentas essenciais...")
executar("sudo apt-get update -qq", check=False)
executar("sudo apt-get install -y -qq curl tar gzip zstd", check=False)

# Tenta instalar o Ollama via o script oficial.
# Para evitar o erro "non-zero exit status 1", vamos capturar o stdout/stderr
# do script para depuração, mesmo que o comando principal falhe.
if shutil.which("ollama") is None:
    print("Tentando instalar Ollama via script oficial com log detalhado...")
    try:
        install_cmd = "curl -fsSL https://ollama.com/install.sh"
        # Baixa o script e executa com bash -x para ver cada comando.
        # Redireciona stderr para stdout para captura completa.
        result = executar(
            f"{install_cmd} | bash -x 2>&1",
            capture=True, check=False
        )

        if result.returncode != 0:
            print("\nAtenção: A instalação do Ollama via script oficial falhou.")
            print("STDOUT/STDERR do script de instalação:")
            print(result.stdout)
            print(result.stderr)
            print("\nTentando instalação manual como fallback...")

            # Fallback para instalação manual se o script falhar
            # A maioria das instâncias Colab usa x86_64
            arch = executar("uname -m", capture=True, check=True).stdout.strip()
            ollama_bin_url = ""
            if arch == "x86_64":
                ollama_bin_url = "https://ollama.com/download/ollama-linux-x86_64"
            elif arch == "aarch64":
                ollama_bin_url = "https://ollama.com/download/ollama-linux-arm64"
            else:
                raise RuntimeError(f"Arquitetura {arch} não suportada para instalação manual do Ollama.")

            print(f"Baixando binário do Ollama para {arch} de {ollama_bin_url}")
            executar(f"curl -L {ollama_bin_url} -o /usr/local/bin/ollama", check=True)
            executar("chmod +x /usr/local/bin/ollama", check=True)
            print("Instalação manual do Ollama concluída.")
        else:
            print("\nInstalação do Ollama via script oficial concluída com sucesso.")
            print(result.stdout) # Print output even if success
    except Exception as e:
        print(f"Erro inesperado durante a instalação do Ollama: {e}")
else:
    print("Ollama já parece estar instalado, pulando a instalação em 4.0.")

# Verificar se Ollama está disponível após as tentativas
if shutil.which("ollama") is not None:
    print("\nOllama binário encontrado no PATH.")
else:
    print("\nAVISO: Ollama não foi encontrado no PATH. Pode haver problemas nas próximas células.")


Atualizando apt-get e instalando ferramentas essenciais...
Ollama já parece estar instalado, pulando a instalação em 4.0.

Ollama binário encontrado no PATH.


In [13]:
# 4.1 — Verificar o binário Ollama instalado

# A instalação robusta já foi feita na célula 4.0 (bfKKgONk2zie).
# Esta célula apenas confirma se o binário está acessível.
if shutil.which("ollama") is None:
    print("Ollama não encontrado. Por favor, execute a célula 4.0 acima.")
else:
    versao = executar(["ollama", "--version"], capture=True, check=False).stdout.strip() \
             or executar(["ollama", "--version"], capture=True, check=False).stderr.strip()
    print(f"Ollama instalado: {versao}")

Ollama instalado: Warning: could not connect to a running Ollama instance


In [14]:
# 4.2 — Iniciar o servidor Ollama com armazenamento no Drive

import requests

os.environ["OLLAMA_MODELS"] = str(OLLAMA_MODELS_DIR)
os.environ["OLLAMA_HOST"] = "127.0.0.1:11434"

OLLAMA_ENV = os.environ.copy()
OLLAMA_LOG = NEXUS / "ollama.log"

def ollama_esta_ativo():
    try:
        r = requests.get("http://127.0.0.1:11434/api/tags", timeout=2)
        return r.ok
    except Exception:
        return False

if not ollama_esta_ativo():
    log_handle = open(OLLAMA_LOG, "ab", buffering=0)
    OLLAMA_PROCESS = subprocess.Popen(
        ["ollama", "serve"],
        env=OLLAMA_ENV,
        stdout=log_handle,
        stderr=subprocess.STDOUT,
    )

    for _ in range(60):
        if ollama_esta_ativo():
            break
        time.sleep(1)
    else:
        raise RuntimeError(
            "Ollama não iniciou. Veja o log em "
            f"{OLLAMA_LOG}"
        )

print("Ollama ativo.")
print(requests.get("http://127.0.0.1:11434/api/tags", timeout=5).json())

Ollama ativo.
{'models': []}


In [15]:
# 4.3 — Baixar somente o que ainda não existe no Drive

MODELO_BASE = "qwen3:4b"
MODELO_EMBED = "nomic-embed-text"

def nomes_modelos_instalados():
    dados = requests.get("http://127.0.0.1:11434/api/tags", timeout=5).json()
    return {m.get("name", "") for m in dados.get("models", [])}

def garantir_modelo(nome):
    instalados = nomes_modelos_instalados()
    if nome in instalados:
        print(f"[OK] {nome} já está persistido.")
        return
    executar(["ollama", "pull", nome], env=OLLAMA_ENV)

garantir_modelo(MODELO_BASE)
garantir_modelo(MODELO_EMBED)

# Variáveis lidas pelos arquivos originais.
os.environ["MODELO"] = MODELO_BASE
os.environ["MODELO_PEQUENO"] = MODELO_BASE
os.environ["MODELO_MEDIO"] = MODELO_BASE
os.environ["MODELO_GRANDE"] = MODELO_BASE

print("Modelos:", sorted(nomes_modelos_instalados()))

Modelos: ['nomic-embed-text:latest', 'qwen3:4b']


In [16]:
# 4.4 — Opcional: baixar os três tamanhos usados originalmente no Dia 5

BAIXAR_MODELOS_DIA5 = False

if BAIXAR_MODELOS_DIA5:
    for nome in ["qwen3:1.7b", "qwen3:4b", "qwen3:8b"]:
        garantir_modelo(nome)

    os.environ["MODELO_PEQUENO"] = "qwen3:1.7b"
    os.environ["MODELO_MEDIO"] = "qwen3:4b"
    os.environ["MODELO_GRANDE"] = "qwen3:8b"

print(
    "papéis:",
    os.environ["MODELO_PEQUENO"],
    os.environ["MODELO_MEDIO"],
    os.environ["MODELO_GRANDE"],
)

papéis: qwen3:4b qwen3:4b qwen3:4b


In [17]:
# 4.5 — Warm-up e confirmação prática do backend

r = executar(
    ["ollama", "run", MODELO_BASE, "Responda apenas com a palavra OK."],
    capture=True, env=OLLAMA_ENV
)
print(r.stdout.strip())

print("\nollama ps:")
print(
    executar(["ollama", "ps"], capture=True, check=False, env=OLLAMA_ENV).stdout
)

print("\nGPU após warm-up:")
executar(["nvidia-smi"], check=False)

Thinking...
Okay, the user wants me to respond with just the word "OK". Let me check th
the instructions again. They said "Responda apenas com a palavra OK." which
which translates to "Respond only with the word OK." So I need to make sure
sure I don't add anything else. Just "OK". No extra spaces, no punctuation.
punctuation. Let me confirm. Yep, the answer should be exactly "OK".
...done thinking.

OK

ollama ps:
NAME        ID              SIZE      PROCESSOR    CONTEXT    UNTIL              
qwen3:4b    359d7dd4bcda    3.2 GB    100% GPU     4096       4 minutes from now    


GPU após warm-up:


CompletedProcess(args=['nvidia-smi'], returncode=0)

## 5. Utilitário para trocar de dia sem colisão de imports

O projeto possui módulos com nomes repetidos (`ferramentas.py`, `agente.py`, `cli.py`). Em um notebook, o cache de `sys.modules` pode fazer o Python reutilizar o módulo de outro dia.

A função abaixo remove explicitamente os módulos didáticos antes de mudar de pasta.

In [18]:
# 5.1 — Isolar imports por dia

MODULOS_DO_CURSO = {
    "agente", "clientes", "ferramentas", "indexar",
    "nexus", "hooks", "steering", "ferramentas_web",
    "equipe", "prompts", "handoff", "memoria_semantica",
    "email_assistente", "app_gradio",
    "avaliacao", "modelos", "roteador", "pipelines", "embeddings_hf",
    "cli",
}

def usar_dia(dia: str):
    alvo = NEXUS / dia
    if not alvo.exists():
        raise ValueError(f"Dia inexistente: {dia}")

    for nome in list(sys.modules):
        if nome in MODULOS_DO_CURSO:
            sys.modules.pop(nome, None)

    prefixos_curso = [str(NEXUS / f"dia{i}") for i in range(1, 6)]
    sys.path[:] = [p for p in sys.path if p not in prefixos_curso]
    sys.path.insert(0, str(alvo))
    os.chdir(NEXUS)
    importlib.invalidate_caches()
    print("Imports ativos:", alvo)

## 6. Validação estática e testes unitários

O material possui **43 funções `test_*`**, que se expandem para **55 casos pytest** por parametrização.

Esses testes são intencionalmente sem LLM e sem rede. Portanto:

- são bons para lógica determinística;
- **não** provam que Ollama, Chroma, LangGraph, Gradio ou os modelos Hugging Face funcionam ponta a ponta;
- os smoke tests de cada dia, abaixo, completam essa lacuna no ambiente Colab.

In [19]:
# 6.1 — Compilação sintática de todo o projeto

r = executar(
    [sys.executable, "-m", "compileall", "-q", str(NEXUS)],
    check=False, capture=True
)
if r.returncode:
    print(r.stdout)
    print(r.stderr)
    raise RuntimeError("Falha de compilação sintática.")
print("Sintaxe Python: OK")

Sintaxe Python: OK


In [20]:
# 6.2 — Executar os 55 casos pytest

r = executar(
    [sys.executable, "-m", "pytest", "-q", str(NEXUS / "testes")],
    check=False, capture=True, cwd=str(NEXUS)
)
print(r.stdout)
if r.stderr:
    print(r.stderr)

if r.returncode != 0:
    raise RuntimeError("Há testes falhando. Não avance sem revisar o erro acima.")

.......................................................                  [100%]
55 passed in 0.89s



# DIA 1 — Harness Python puro + tool calling

### Código original

- `dia1/agente.py`
- `dia1/clientes.py`
- `dia1/ferramentas.py`

### Pontos fortes

- loop de ferramentas explícito e didático;
- argumentos de tool call validados como JSON objeto;
- exceções de ferramenta viram observação para o agente, em vez de derrubar todo o loop;
- calculadora usa AST, não `eval`;
- a versão atual usa `Path.relative_to()` para impedir leitura fora de `docs/`.

### Observação de manutenção

O arquivo atual ainda contém grandes blocos de uma implementação antiga comentada. Git já preserva histórico; esses blocos aumentam ruído sem contribuir para o laboratório.

In [21]:
# Dia 1.1 — Testar ferramentas sem LLM

usar_dia("dia1")
import ferramentas as f1

print("950 * 24 =", f1.calcular("950 * 24"))
print("\nArquivos disponíveis:")
print(f1.listar_arquivos())

print("\nTeste de path traversal:")
print(f1.ler_arquivo("../../etc/passwd"))

Imports ativos: /content/minicurso-mult-agents/codigo/nexus/dia1
950 * 24 = 22800

Arquivos disponíveis:
2023_relatorio.md
2024_relatorio.md
estatuto.md
fornecedores.md
notas_reuniao.md

Teste de path traversal:
ERRO: caminho fora da pasta permitida. Use apenas arquivos existentes em docs/.


In [22]:
# Dia 1.2 — Executar o agente pelo endpoint OpenAI-compatible do Ollama

usar_dia("dia1")
import agente as agente1

resposta_d1 = agente1.rodar(
    "Qual foi o faturamento de 2024? Leia o documento necessário e cite o nome dele.",
    backend="ollama",
    max_passos=8,
    verbose=True,
)

print("\nRESPOSTA DIA 1\n", resposta_d1)
persistir_estado()

Imports ativos: /content/minicurso-mult-agents/codigo/nexus/dia1
[passo 0] listar_arquivos({}) -> 2023_relatorio.md 2024_relatorio.md estatuto.md fornecedores.md notas_reuniao.md | 1437 tokens
[passo 1] ler_arquivo({'caminho': '2024_relatorio.md'}) -> # Relatório Anual 2024 — Cooperativa Vale Verde  ## Resumo executivo  Faturamento bruto de R$ 5.640.000,00 em 2024. O nú | 3584 tokens
[passo 2] resposta final | 6367 tokens | 33.1s

RESPOSTA DIA 1
 O faturamento de 2024 foi de R$ 5.640.000,00. O documento utilizado foi **2024_relatorio.md**.
Estado persistido em /content/drive/MyDrive/minicurso-mult-agents-colab/state


# DIA 2 — LangChain + RAG + Chroma

### Código original

- `dia2/agente.py`
- `dia2/ferramentas.py`
- `dia2/indexar.py`

### Mudança conceitual

O loop passa a usar `ChatOllama`, tools LangChain e um índice Chroma com `nomic-embed-text`.

### Pontos que exigem atenção

1. `dia2/ferramentas.py` usa comparação textual com `startswith()` para validar caminhos. Isso é inferior ao `Path.relative_to()` usado no Dia 1.
2. O índice `.chroma` não contém fingerprint dos documentos/modelo/chunking. Se `docs/` mudar, o índice pode ficar obsoleto até ser recriado.
3. A persistência original é local. Neste notebook `.chroma` é restaurado e copiado para o Drive.

In [23]:
# Dia 2.1 — Criar/reutilizar o índice Chroma

usar_dia("dia2")
import indexar as indexar2

recriar = not (NEXUS / ".chroma").exists()
banco_d2 = indexar2.construir(recriar=recriar)

for d in banco_d2.similarity_search("faturamento de 2024", k=2):
    print("\n---", Path(d.metadata.get("source", "?")).name)
    print(d.page_content[:500])

persistir_estado()

Imports ativos: /content/minicurso-mult-agents/codigo/nexus/dia2
5 documentos -> 7 trechos

--- 2024_relatorio.md
# Relatório Anual 2024 — Cooperativa Vale Verde

## Resumo executivo

Faturamento bruto de R$ 5.640.000,00 em 2024. O número de cooperados chegou a 261.
A linha de café especial passou a responder por mais da metade da receita.

## Faturamento por linha

| Linha | Faturamento (R$) | Participação |
|---|---|---|
| Café especial | 3.102.000 | 55% |
| Hortifrúti | 1.410.000 | 25% |
| Laticínios | 1.128.000 | 20% |

## Custos

Custo operacional total: R$ 4.230.000,00. Margem operacional de 25%.
Logí

--- 2023_relatorio.md
# Relatório Anual 2023 — Cooperativa Vale Verde

## Resumo executivo

O exercício de 2023 encerrou com faturamento bruto de R$ 4.820.000,00, crescimento de
12% sobre 2022. O número de cooperados subiu de 214 para 238.

## Faturamento por linha

| Linha | Faturamento (R$) | Participação |
|---|---|---|
| Café especial | 2.410.000 | 50% |
| Hortifrúti | 1.446.00

In [24]:
# Dia 2.2 — Executar o agente RAG

usar_dia("dia2")
import agente as agente2

resposta_d2 = agente2.rodar(
    "Qual foi o faturamento de 2024? Cite a fonte.",
    max_passos=8,
    verbose=True,
)
print("\nRESPOSTA DIA 2\n", resposta_d2)

persistir_estado()

Imports ativos: /content/minicurso-mult-agents/codigo/nexus/dia2
[passo 0] listar_arquivos({}) -> - 2023_relatorio.md (707 bytes)
- 2024_relatorio.md (713 bytes)
- estatuto.md (1
[passo 1] buscar_documentos({'consulta': 'faturamento 2024', 'k': 1}) -> [fonte: 2024_relatorio.md]
# Relatório Anual 2024 — Cooperativa Vale Verde

## R

RESPOSTA DIA 2
 O faturamento de 2024 foi de R$ 5.640.000,00 [fonte: 2024_relatorio.md].
Estado persistido em /content/drive/MyDrive/minicurso-mult-agents-colab/state


# DIA 3 — LangGraph + memória + HITL + constraints/steering

### Código original

- `dia3/nexus.py`
- `dia3/cli.py`
- `dia3/hooks.py`
- `dia3/steering.py`
- `dia3/ferramentas_web.py`

### Achado estrutural importante

`hooks.py` e `steering.py` existem e possuem testes, mas **não estão conectados ao grafo construído em `nexus.py`**. O grafo principal usa `ToolNode(ferramentas)` diretamente.

Logo, hoje há diferença entre:

```text
conteúdo didático presente no repositório
≠
comportamento efetivamente executado pelo grafo principal
```

O notebook testa as políticas separadamente e executa o grafo principal sem fingir que os hooks já estão integrados.

### Segurança da web

`ler_pagina()` aceita qualquer URL `http(s)`. Antes de usar essa ferramenta em um agente real, adicione proteção contra SSRF, endereços locais/metadata e conteúdo não confiável.

In [25]:
# Dia 3.1 — Constraints e steering como unidades puras

usar_dia("dia3")
import hooks, steering

print(
    "Path traversal:",
    hooks.avaliar_politicas(
        "ler_arquivo",
        {"caminho": "../../etc/passwd"},
        {}
    )
)

print(
    "Comando destrutivo:",
    hooks.avaliar_politicas(
        "shell",
        {"comando": "rm -rf /"},
        {}
    )
)

print(
    "Estagnação:",
    steering.detectar_estagnacao({"passos": 7, "achados": []})
)

Imports ativos: /content/minicurso-mult-agents/codigo/nexus/dia3
Path traversal: leitura fora de docs/ nao e permitida
Comando destrutivo: comando destrutivo bloqueado pela politica de seguranca
Estagnação: True


In [31]:
# Dia 3.2 — Executar o grafo LangGraph principal
usar_dia("dia3")
from langchain_core.messages import HumanMessage
import nexus as nexus3
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver

db_path = str(NEXUS / "nexus_colab.db")

# Abrimos a conexão fora para garantir controle total sobre o fechamento
conn = sqlite3.connect(db_path, check_same_thread=False)
try:
    checkpointer = SqliteSaver(conn)
    # Compilamos o grafo garantindo o uso do checkpointer ativo
    grafo_d3 = nexus3.compilar(
        caminho_db=db_path,
        aprovar_ferramentas=False,
    )
    # Forçamos o grafo a usar o nosso checkpointer com a conexão aberta
    grafo_d3.checkpointer = checkpointer

    config_d3 = {
        "configurable": {"thread_id": "colab-dia3"},
        "recursion_limit": 30,
    }

    estado_d3 = grafo_d3.invoke(
        {
            "messages": [HumanMessage("Qual foi o faturamento de 2024? Cite a fonte.")],
            "passos": 0,
        },
        config_d3,
    )
    print(estado_d3["messages"][-1].content)
finally:
    conn.close()

persistir_estado()

Imports ativos: /content/minicurso-mult-agents/codigo/nexus/dia3
O faturamento de 2024 foi de R$ 5.640.000,00 [fonte: 2024_relatorio.md].
Estado persistido em /content/drive/MyDrive/minicurso-mult-agents-colab/state


In [33]:
# Dia 3.3 — Ferramentas web são opcionais porque enviam a consulta para fora da máquina

EXECUTAR_WEB = False

if EXECUTAR_WEB:
    usar_dia("dia3")
    import ferramentas_web as fw

    print(
        fw.buscar_web.invoke({
            "consulta": "LangGraph state graph",
            "k": 3,
        })
    )
else:
    print("Busca web não executada. Defina EXECUTAR_WEB=True para testar.")

Busca web não executada. Defina EXECUTAR_WEB=True para testar.


# DIA 4 — Equipe multiagente + memória + e-mail + Gradio

### Código original

- `dia4/equipe.py`
- `dia4/prompts.py`
- `dia4/handoff.py`
- `dia4/memoria_semantica.py`
- `dia4/email_assistente.py`
- `dia4/app_gradio.py`

### Achados de integração

- `handoff.py` demonstra handoff, mas a equipe principal usa supervisor e **não usa esses handoffs**.
- `memoria_semantica.py` usa `InMemoryStore`; a função/tool `lembrar()` retorna `"Anotado"` mas não grava no store. É um placeholder didático, não memória persistente.
- `email_assistente.processar()` chama `construir_grafo()` sem `retriever`; portanto a rota FAQ padrão trabalha com `(base vazia)`. A célula abaixo passa explicitamente o índice do Dia 2.
- `MAX_RODADAS` é incrementado no crítico, não em toda transição. O `recursion_limit` usado neste notebook fornece uma segunda barreira contra ciclos.

In [32]:
# Dia 4.1 — Executar a equipe multiagente
usar_dia("dia4")
from langchain_core.messages import HumanMessage
import equipe as equipe4
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver

db_path_d4 = str(NEXUS / "equipe_colab.db")
pergunta_d4 = "Compare os fornecedores disponíveis e recomende um, citando fontes."

conn_d4 = sqlite3.connect(db_path_d4, check_same_thread=False)
try:
    checkpointer_d4 = SqliteSaver(conn_d4)
    grafo_d4 = equipe4.compilar(caminho_db=db_path_d4)
    grafo_d4.checkpointer = checkpointer_d4

    entrada_d4 = {
        "messages": [HumanMessage(pergunta_d4)],
        "pergunta": pergunta_d4,
        "proximo": "",
        "instrucao": pergunta_d4,
        "achados": [],
        "rascunho": "",
        "veredito": "",
        "rodadas": 0,
    }

    config_d4 = {
        "configurable": {"thread_id": "colab-dia4"},
        "recursion_limit": 40,
    }

    final_d4 = grafo_d4.invoke(entrada_d4, config_d4)
    print("VEREDITO:", final_d4.get("veredito"))
    print("\nRELATÓRIO:\n")
    print(final_d4.get("rascunho") or final_d4["messages"][-1].content)
finally:
    conn_d4.close()

persistir_estado()

Imports ativos: /content/minicurso-mult-agents/codigo/nexus/dia4
VEREDITO: aprovado

RELATÓRIO:

# Relatório de Comparação entre Verde Embalagens Ltda. e Vale Embalagens S.A.

## Resumo
Verde Embalagens Ltda. apresenta mensalidade de R$ 4.500,00, prazo de entrega de 7 dias e contrato até 12/2026, enquanto Vale Embalagens S.A. cobra R$ 6.200,00 com prazo de 12 dias e contrato até 06/2026. Ambos os contratos estão dentro do prazo de 24 meses, evitando necessidade de aprovação do Conselho de Administração. O conselho destacou que o preço unitário não é suficiente para comparação, enfatizando a diferença significativa nas mensalidades.

## Achados
- Verde Embalagens Ltda.: CNPJ 11.222.333/0001-44, preço unitário da caixa padrão R$ 3,20, mensalidade de contrato R$ 4.500,00, prazo de entrega 7 dias, contrato vigente até 12/2026 [fonte: fornecedores.md]
- Vale Embalagens S.A.: CNPJ 55.666.777/0001-88, preço unitário da caixa padrão R$ 2,95, mensalidade de contrato R$ 6.200,00, prazo de entreg

In [34]:
# Dia 4.2 — Memória semântica: demonstração do código original

usar_dia("dia4")
import memoria_semantica as ms4

memoria = ms4.criar_memoria()
ms4.guardar(
    memoria,
    "aluno",
    "O aluno prefere respostas técnicas com fontes."
)

print(
    ms4.lembrancas_de(
        memoria,
        "aluno",
        "Como devo responder para este aluno?",
        limite=1,
    )
)

Imports ativos: /content/minicurso-mult-agents/codigo/nexus/dia4
- O aluno prefere respostas técnicas com fontes.


In [45]:
# Dia 4.3 — Assistente de e-mail com o retriever explicitamente conectado

# Reutiliza o banco do Dia 2.
usar_dia("dia2")
import indexar as indexar_email
banco_email = indexar_email.construir(recriar=False)

usar_dia("dia4")
import email_assistente as email4
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.types import Command

db_path_email = str(NEXUS / "email_colab.db")

# Garantimos que a conexão do checkpointer esteja aberta durante o ciclo do grafo
conn_email = sqlite3.connect(db_path_email, check_same_thread=False)
try:
    checkpointer_email = SqliteSaver(conn_email)
    grafo_email = email4.construir_grafo(retriever=banco_email)
    grafo_email.checkpointer = checkpointer_email

    emails = json.loads(
        (NEXUS / "dados" / "emails.json").read_text(encoding="utf-8")
    )
    email_exemplo = emails[0]
    config_email = {
        "configurable": {"thread_id": "colab-email-0"}
    }

    estado_email = grafo_email.invoke(email_exemplo, config_email)

    snapshot = grafo_email.get_state(config_email)
    if snapshot.tasks and snapshot.tasks[0].interrupts:
        pendente = snapshot.tasks[0].interrupts[0].value
        print("Rascunho aguardando humano:\n", pendente)
        # Smoke test seguro: NÃO envia nada.
        estado_email = grafo_email.invoke(
            Command(resume={"acao": "descartar"}),
            config_email,
        )

    print("Estado final:", estado_email)
finally:
    conn_email.close()

persistir_estado()

Imports ativos: /content/minicurso-mult-agents/codigo/nexus/dia2
Imports ativos: /content/minicurso-mult-agents/codigo/nexus/dia4
Estado final: {'remetente': 'marina.alves@cooperativavaleverde.com.br', 'assunto': 'Prazo de troca do equipamento', 'corpo': 'Bom dia. Comprei o modelo VX-200 ha doze dias e ele nao atende ao que precisamos. Qual e o prazo para solicitar a troca e o que preciso enviar junto?', 'intencao': 'spam', 'urgencia': 'baixa', 'resumo': 'Cliente solicita prazo para troca de equipamento VX-200 adquirido há 12 dias, informando que não atende às necessidades. Não há menção a prejuízo em curso ou danos imediatos, portanto, urgência baixa.', 'rascunho': '', 'rota': 'arquivar', 'decisao': 'arquivado'}
Estado persistido em /content/drive/MyDrive/minicurso-mult-agents-colab/state


In [36]:
# Dia 4.4 — Gradio no Colab (opcional)
#
# ATENÇÃO: share=True cria um link público temporário. Não exponha documentos
# confidenciais nem ferramentas perigosas. A aplicação original usa share=False.

EXECUTAR_GRADIO_PUBLICO = False

if EXECUTAR_GRADIO_PUBLICO:
    usar_dia("dia4")
    import app_gradio
    app_gradio.app.launch(share=True, debug=False)
else:
    print(
        "Gradio público desativado. "
        "Defina EXECUTAR_GRADIO_PUBLICO=True somente se os dados forem apropriados."
    )

Gradio público desativado. Defina EXECUTAR_GRADIO_PUBLICO=True somente se os dados forem apropriados.


# DIA 5 — Hugging Face + modelos por papel + avaliação

### Código original

- `dia5/modelos.py`
- `dia5/embeddings_hf.py`
- `dia5/roteador.py`
- `dia5/pipelines.py`
- `dia5/avaliacao.py`
- `dia5/cli.py`

### Achados críticos

1. `dia5/cli.py` continua importando e executando `equipe.compilar` do Dia 4. Ele **não usa** `modelos.py`, `embeddings_hf.py` nem `roteador.py`.
2. `dia5/avaliacao.py` faz `from cli import responder as nexus`, mas **`dia5/cli.py` não define `responder`**. O entrypoint de avaliação, como está, falha.
3. `embeddings_hf.py` fixa `device="cpu"`.
4. Os `transformers.pipeline(...)` em `pipelines.py` e `roteador.py` não passam `device`, portanto não utilizam explicitamente a GPU.
5. O chat de `pipelines.py` carrega o modelo sem `device_map`, logo também tende a permanecer em CPU.

Nesta adaptação Colab, as células GPU abaixo reutilizam os mesmos modelos/tarefas, mas configuram CUDA explicitamente. A avaliação usa um adapter do grafo do Dia 4 para contornar o `responder` ausente sem esconder o defeito do código original.

In [37]:
# Dia 5.1 — GPU Hugging Face e modelos por papel

import torch

if not torch.cuda.is_available():
    raise RuntimeError("PyTorch não está vendo CUDA.")

HF_DEVICE_PIPELINE = 0
HF_DEVICE_NAME = "cuda"

print("torch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))

usar_dia("dia5")
import modelos as modelos5

print("Modelos por papel configurados:")
for papel in ["supervisor", "pesquisador", "analista", "redator", "critico"]:
    print(" -", papel, "->", modelos5.para(papel).model)

torch: 2.11.0+cu128
CUDA: 12.8
GPU: Tesla T4
Imports ativos: /content/minicurso-mult-agents/codigo/nexus/dia5
Modelos por papel configurados:
 - supervisor -> qwen3:4b
 - pesquisador -> qwen3:4b
 - analista -> qwen3:4b
 - redator -> qwen3:4b
 - critico -> qwen3:4b


In [38]:
# Dia 5.2 — Embedding Hugging Face em GPU, com cache persistente

from langchain_huggingface import HuggingFaceEmbeddings

MODELO_EMB_HF = "intfloat/multilingual-e5-small"

emb_hf_gpu = HuggingFaceEmbeddings(
    model_name=MODELO_EMB_HF,
    model_kwargs={"device": HF_DEVICE_NAME},
    encode_kwargs={"normalize_embeddings": True},
)

vetor = emb_hf_gpu.embed_query("faturamento da cooperativa em 2024")
print("Dimensão:", len(vetor))
print("Dispositivo solicitado:", HF_DEVICE_NAME)

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/498k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Dimensão: 384
Dispositivo solicitado: cuda


In [39]:
# Dia 5.3 — Índice Chroma com embedding HF em GPU

from langchain_chroma import Chroma
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

PERSIST_HF_GPU = NEXUS / ".chroma_hf_gpu"

if PERSIST_HF_GPU.exists():
    banco_hf_gpu = Chroma(
        persist_directory=str(PERSIST_HF_GPU),
        embedding_function=emb_hf_gpu,
    )
else:
    docs_hf = DirectoryLoader(
        str(NEXUS / "docs"),
        glob="**/*.md",
        loader_cls=TextLoader,
        loader_kwargs={"encoding": "utf-8"},
    ).load()

    partes_hf = RecursiveCharacterTextSplitter(
        chunk_size=800,
        chunk_overlap=120,
        separators=["\n## ", "\n### ", "\n\n", "\n", " "],
    ).split_documents(docs_hf)

    banco_hf_gpu = Chroma.from_documents(
        partes_hf,
        emb_hf_gpu,
        persist_directory=str(PERSIST_HF_GPU),
    )

for d in banco_hf_gpu.similarity_search("faturamento de 2024", k=2):
    print("\n---", Path(d.metadata.get("source", "?")).name)
    print(d.page_content[:400])

persistir_estado()


--- notas_reuniao.md
# Notas da reunião do Conselho — 14 de março de 2025

Presentes: 7 dos 9 conselheiros.

## Pauta 1 — Armazenagem

O investimento em armazenagem aprovado em 2024 teve o orçamento revisto. O valor
atualizado ficou em um milhão e duzentos mil reais, bem acima do previsto
originalmente. O conselho pediu três novas cotações antes de decidir.

## Pauta 2 — Embalagens

Discutida a troca de fornecedor de 

--- 2024_relatorio.md
# Relatório Anual 2024 — Cooperativa Vale Verde

## Resumo executivo

Faturamento bruto de R$ 5.640.000,00 em 2024. O número de cooperados chegou a 261.
A linha de café especial passou a responder por mais da metade da receita.

## Faturamento por linha

| Linha | Faturamento (R$) | Participação |
|---|---|---|
| Café especial | 3.102.000 | 55% |
| Hortifrúti | 1.410.000 | 25% |
| Laticínios | 1.1
Estado persistido em /content/drive/MyDrive/minicurso-mult-agents-colab/state


In [40]:
# Dia 5.4 — Rerank em GPU

from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1",
    device=HF_DEVICE_NAME,
)

consulta = "custo total dos fornecedores em 24 meses"
candidatos = banco_hf_gpu.similarity_search(consulta, k=10)
scores = reranker.predict(
    [(consulta, d.page_content) for d in candidatos]
)

ordenados = [
    d for _, d in sorted(
        zip(scores, candidatos),
        key=lambda x: -float(x[0]),
    )
]

for d in ordenados[:4]:
    print("\n---", Path(d.metadata.get("source", "?")).name)
    print(d.page_content[:350])

config.json:   0%|          | 0.00/891 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/435 [00:00<?, ?B/s]

...

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]


--- notas_reuniao.md
# Notas da reunião do Conselho — 14 de março de 2025

Presentes: 7 dos 9 conselheiros.

## Pauta 1 — Armazenagem

O investimento em armazenagem aprovado em 2024 teve o orçamento revisto. O valor
atualizado ficou em um milhão e duzentos mil reais, bem acima do previsto
originalmente. O conselho pediu três novas cotações antes de decidir.

## Pauta 2

--- 2023_relatorio.md
# Relatório Anual 2023 — Cooperativa Vale Verde

## Resumo executivo

O exercício de 2023 encerrou com faturamento bruto de R$ 4.820.000,00, crescimento de
12% sobre 2022. O número de cooperados subiu de 214 para 238.

## Faturamento por linha

| Linha | Faturamento (R$) | Participação |
|---|---|---|
| Café especial | 2.410.000 | 50% |
| Hortifrút

--- fornecedores.md
# Fornecedores homologados — Cooperativa Vale Verde

## Embalagens

### Verde Embalagens Ltda.
- CNPJ: 11.222.333/0001-44
- Preço unitário da caixa padrão: R$ 3,20
- Mensalidade de contrato: R$ 4.500,00
- Prazo de entrega: 7 dias
- 

In [41]:
# Dia 5.5 — Pipelines Transformers usando GPU explicitamente

usar_dia("dia5")
import pipelines as p5
from transformers import pipeline

sentimento_gpu = pipeline(
    "sentiment-analysis",
    model=p5.MODELO_SENTIMENTO,
    device=HF_DEVICE_PIPELINE,
)

print(
    sentimento_gpu(
        ["O equipamento é excelente.", "O atendimento foi péssimo."],
        truncation=True,
    )
)

zeroshot_gpu = pipeline(
    "zero-shot-classification",
    model=p5.MODELO_ZEROSHOT,
    device=HF_DEVICE_PIPELINE,
)

print(
    zeroshot_gpu(
        "Preciso comprar um torno industrial.",
        candidate_labels=p5.CATEGORIAS,
    )
)

Imports ativos: /content/minicurso-mult-agents/codigo/nexus/dia5


config.json:   0%|          | 0.00/953 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  669MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 16.3MB            

tokenizer.json: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/23.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/286 [00:00<?, ?B/s]

{'sequence': 'Preciso comprar um torno industrial.', 'labels': ['equipamento industrial', 'servico', 'insumo quimico', 'pecas de reposicao', 'ferramenta manual', 'maquinario agricola'], 'scores': [0.9047704935073853, 0.05875089392066002, 0.030914226546883583, 0.0034163743257522583, 0.0015443984884768724, 0.0006036150152795017]}


In [50]:
# Dia 5.6 — Chat Transformers pequeno em GPU

from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

MODELO_CHAT_HF = "Qwen/Qwen3-0.6B"

tok_chat = AutoTokenizer.from_pretrained(MODELO_CHAT_HF)
modelo_chat = AutoModelForCausalLM.from_pretrained(
    MODELO_CHAT_HF,
    torch_dtype="auto",
    device_map="auto",
)

mensagens_chat = [
    {
        "role": "system",
        "content": "Responda em português e de forma objetiva.",
    },
    {
        "role": "user",
        "content": "Explique em duas frases a função de um agente supervisor.",
    },
]

# Preparação das entradas
inputs = tok_chat.apply_chat_template(
    mensagens_chat,
    add_generation_prompt=True,
    return_tensors="pt",
)

# Movemos as entradas para a GPU
# o do_sample para True para introduzir variabilidade e tentar contornar a interrupção da frase,
# mas isso pode tornar a resposta menos objetiva.
inputs = {k: v.to(modelo_chat.device) for k, v in inputs.items()}

with torch.inference_mode():
    saida_chat = modelo_chat.generate(
        **inputs,
        max_new_tokens=250,  # Aumentado para não cortar a frase
        do_sample=False,     #
        pad_token_id=tok_chat.eos_token_id,
    )

# Decodifica apenas a parte gerada (removendo o prompt)
input_ids_len = inputs["input_ids"].shape[-1]
novo = saida_chat[0][input_ids_len:]
print(tok_chat.decode(novo, skip_special_tokens=True).strip())

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

<think>
Okay, the user wants me to explain the function of a supervisory agent in two sentences. Let me start by recalling what a supervisory agent does. They probably manage or guide a team or process. First sentence: maybe they set goals and ensure tasks are done. Second sentence: maybe they provide feedback and support. Need to make sure it's clear and concise. Let me check if I'm using the right terms. Supervisory agent, not supervisor. Yes, that's correct. Alright, that should do it.
</think>

Um agente supervisor tem a função de orientar e guiar um processo ou equipe, garantindo que as metas sejam alcançadas com eficiência. O outro aspecto é fornecer feedback contínuo para ajustar estratégias e resolver problemas.


## Dia 5.7 — Avaliação completa

O código original de `dia5/avaliacao.py` espera uma função:

```python
responder(pergunta) -> (resposta, fontes, tokens)
```

mas essa função não existe em `dia5/cli.py`.

O adapter abaixo fornece esse contrato usando a equipe do Dia 4. Ele não altera o arquivo original e deixa a incompatibilidade explícita.

In [47]:
# Dia 5.7a — Adapter para a interface exigida por avaliacao.py

usar_dia("dia4")
import equipe as equipe_av

grafo_av = equipe_av.compilar(
    caminho_db=str(NEXUS / "equipe_avaliacao_colab.db")
)

def responder_para_avaliacao(pergunta: str):
    from langchain_core.messages import HumanMessage

    entrada = {
        "messages": [HumanMessage(pergunta)],
        "pergunta": pergunta,
        "proximo": "",
        "instrucao": pergunta,
        "achados": [],
        "rascunho": "",
        "veredito": "",
        "rodadas": 0,
    }

    config = {
        "configurable": {
            "thread_id": f"eval-{uuid.uuid4()}"
        },
        "recursion_limit": 40,
    }

    final = grafo_av.invoke(entrada, config)
    resposta = (
        final.get("rascunho")
        or final["messages"][-1].content
        or ""
    )

    fontes = sorted(set(
        re.findall(r"\[fonte:\s*([^\]]+)\]", resposta, flags=re.I)
    ))

    # O grafo atual não agrega usage metadata de todos os nós.
    tokens = 0
    return resposta, fontes, tokens

print("Adapter pronto.")

Imports ativos: /content/minicurso-mult-agents/codigo/nexus/dia4
Adapter pronto.


In [48]:
# Dia 5.7b — Rodar os 10 casos (opcional: várias chamadas locais ao LLM)

RODAR_AVALIACAO_COMPLETA = False

usar_dia("dia5")
import avaliacao as avaliacao5

casos = avaliacao5.carregar_casos(
    NEXUS / "avaliacao" / "casos.jsonl"
)

if RODAR_AVALIACAO_COMPLETA:
    linhas = avaliacao5.rodar(
        responder_para_avaliacao,
        casos,
    )
    print(avaliacao5.tabela(linhas))

    destino = NEXUS / "saida" / "avaliacao_colab.json"
    destino.parent.mkdir(parents=True, exist_ok=True)
    destino.write_text(
        json.dumps(
            {
                "rotulo": "colab",
                "resumo": avaliacao5.resumir(linhas),
                "linhas": linhas,
            },
            ensure_ascii=False,
            indent=2,
        ),
        encoding="utf-8",
    )
    print("Salvo em:", destino)
else:
    print(
        f"{len(casos)} casos carregados. "
        "Defina RODAR_AVALIACAO_COMPLETA=True para executar todos."
    )

persistir_estado()

Imports ativos: /content/minicurso-mult-agents/codigo/nexus/dia5
10 casos carregados. Defina RODAR_AVALIACAO_COMPLETA=True para executar todos.
Estado persistido em /content/drive/MyDrive/minicurso-mult-agents-colab/state


# 7. Persistência final e inspeção

Execute esta célula antes de encerrar o runtime.

In [52]:
# 7.1 — Persistir estado final e mostrar consumo

persistir_estado()

print("\nDrive:")
executar(["du", "-sh", str(DRIVE_ROOT)], check=False)

print("\nModelos Ollama:")
executar(["du", "-sh", str(OLLAMA_MODELS_DIR)], check=False)

print("\nCache Hugging Face:")
executar(["du", "-sh", str(HF_ROOT)], check=False)

print("\nArquivos de estado:")
for p in sorted(DRIVE_STATE.rglob("*")):
    if p.is_file():
        print(p.relative_to(DRIVE_STATE), p.stat().st_size)

Estado persistido em /content/drive/MyDrive/minicurso-mult-agents-colab/state

Drive:

Modelos Ollama:

Cache Hugging Face:

Arquivos de estado:
.chroma/chroma.sqlite3 290816
.chroma/e233db78-c3b7-44d2-b229-77191c3d4ca4/data_level0.bin 321200
.chroma/e233db78-c3b7-44d2-b229-77191c3d4ca4/header.bin 100
.chroma/e233db78-c3b7-44d2-b229-77191c3d4ca4/length.bin 400
.chroma/e233db78-c3b7-44d2-b229-77191c3d4ca4/link_lists.bin 0
.chroma_hf_gpu/2f0cff25-60c4-4072-9cfd-cbe7a445f4b0/data_level0.bin 167600
.chroma_hf_gpu/2f0cff25-60c4-4072-9cfd-cbe7a445f4b0/header.bin 100
.chroma_hf_gpu/2f0cff25-60c4-4072-9cfd-cbe7a445f4b0/length.bin 400
.chroma_hf_gpu/2f0cff25-60c4-4072-9cfd-cbe7a445f4b0/link_lists.bin 0
.chroma_hf_gpu/chroma.sqlite3 286720
REVISAO_TECNICA_E_COLAB.md 9711
email_colab.db 28672
equipe_avaliacao_colab.db 0
equipe_colab.db 253952
nexus_colab.db 53248
ollama.log 228777


# 8. Sincronizar alterações de código para o clone persistente no Drive

Esta função é **intencionalmente explícita**. Ela não faz commit nem push no GitHub. Serve apenas para persistir no Drive mudanças feitas manualmente nos `.py`, `.md`, `.json`, `.csv` etc. durante o laboratório.

Não sincroniza estados gerados, caches, bancos ou `__pycache__` para dentro do repositório.

In [53]:
# 8.1 — Opcional: persistir edições de código feitas em /content

def persistir_codigo_no_drive():
    ignorar = shutil.ignore_patterns(
        ".git", "__pycache__", "*.pyc",
        ".chroma", ".chroma_hf", ".chroma_hf_gpu",
        "*.db", "*.db-wal", "*.db-shm",
        "saida", "tracos", "ollama.log"
    )

    for item in RUNTIME_REPO.iterdir():
        if item.name == ".git":
            continue
        destino = DRIVE_REPO / item.name
        if item.is_dir():
            shutil.copytree(
                item,
                destino,
                dirs_exist_ok=True,
                ignore=ignorar,
            )
        else:
            shutil.copy2(item, destino)

    print("Código sincronizado para:", DRIVE_REPO)

# Descomente quando tiver editado arquivos-fonte em /content:
# persistir_codigo_no_drive()

# 9. Matriz de validação recomendada

| Camada | Dia 1 | Dia 2 | Dia 3 | Dia 4 | Dia 5 |
|---|---|---|---|---|---|
| sintaxe Python | automático | automático | automático | automático | automático |
| pytest determinístico | automático | automático | automático | automático | automático |
| Ollama responde | sim | sim | sim | sim | sim |
| tool calling | sim | sim | via grafo | via agentes | conforme módulo |
| RAG | leitura direta | Chroma/Ollama | herdado Dia 2 | herdado Dia 2 | HF GPU opcional |
| memória | histórico | histórico | SQLite | SQLite + store em memória | avaliação/traces |
| GPU Ollama | sim | sim | sim | sim | sim |
| GPU Hugging Face | — | — | — | — | explícita no notebook |
| persistência Drive | código/modelos | + Chroma | + SQLite | + SQLite/saídas | + HF/avaliação |

## Critério para considerar Dias 2–5 “validados”

Não basta o `pytest` passar. Registre, no mínimo:

1. modelo e GPU usados;
2. resultado de `ollama ps`;
3. tempo de criação/reabertura dos índices;
4. uma execução de sucesso por dia;
5. uma execução de erro/recusa;
6. o resultado dos 10 casos de `avaliacao/casos.jsonl`;
7. versão das bibliotecas (`pip freeze`);
8. commit Git do código executado.